In [ ]:
from manim import *

import icm_anim as anim

In [ ]:
class CircularBuffer(Scene):
    """A feedback delay line, y[n] = x[n] + y[n - M], stored in a circular
    buffer of M = 3 samples. Each step reads the oldest sample at the gold
    pointer, adds the input, overwrites the oldest sample with the result,
    and advances the pointer, wrapping around at the end. No sample ever
    moves, and the output repeats every M samples."""

    def construct(self):
        M = 3
        x = [1, 2, 3, 0, 0, 0, 0, 0, 0]
        N = len(x)
        values = [0] * M                  # the buffer starts out silent

        rule = MathTex(r"y[n] = x[n] + y[n - 3]").scale(0.8).move_to([0, 3.35, 0])

        # the buffer: M cells in a row, indices above, a wrap arrow over the top
        cells = VGroup(*[Square(side_length=1.15, color=anim.IRON, stroke_width=2.5)
                         for _ in range(M)]).arrange(RIGHT, buff=0).move_to([0, 1.3, 0])
        nums = [DecimalNumber(0, num_decimal_places=0).scale(0.9).move_to(c)
                for c in cells]
        idx = VGroup(*[MathTex(str(m)).scale(0.5).next_to(cells[m], UP, buff=0.12)
                       for m in range(M)])
        wrap = CurvedArrow(cells[-1].get_top() + [0.25, 0.5, 0],
                           cells[0].get_top() + [-0.25, 0.5, 0],
                           angle=0.5, color=anim.IRON, stroke_width=2, tip_length=0.16)
        buf_label = Text("circular buffer", font_size=24).next_to(cells, LEFT, buff=0.45)

        # the pointer to the oldest sample, y[n - 3], and its highlight
        arrow = Arrow([0, -0.75, 0], [0, 0.15, 0], buff=0, color=anim.GOLD,
                      stroke_width=5, max_tip_length_to_length_ratio=0.3)
        i_label = MathTex("i", color=anim.GOLD).scale(0.7).next_to(arrow, RIGHT, buff=0.1)
        pointer = VGroup(arrow, i_label)
        pointer.next_to(cells[0], DOWN, buff=0.08)
        pointer.shift((cells[0].get_center()[0] - arrow.get_center()[0]) * RIGHT)
        hl = Square(side_length=1.15, color=anim.GOLD, stroke_width=6).move_to(cells[0])

        # the output, one red stem per step
        ax = Axes(x_range=[0, N, 1], y_range=[0, 3.5, 1], x_length=10.0, y_length=1.9,
                  axis_config={"color": anim.IRON, "include_ticks": False,
                               "stroke_width": 1.5},
                  tips=False).move_to([0, -2.75, 0])
        y_label = MathTex("y[n]", color=anim.RED).scale(0.6).next_to(ax.c2p(0, 3.5), LEFT, buff=0.2)
        n_label = MathTex("n").scale(0.55).next_to(ax.c2p(N, 0), RIGHT, buff=0.15)

        self.play(FadeIn(rule), FadeIn(cells), *[FadeIn(v) for v in nums], FadeIn(idx),
                  FadeIn(wrap), FadeIn(buf_label), FadeIn(ax), FadeIn(y_label),
                  FadeIn(n_label), run_time=1.0)
        self.play(FadeIn(pointer), FadeIn(hl), run_time=0.5)

        calc = None
        stems = VGroup()
        self.add(stems)
        for n in range(N):
            m = n % M
            old, new = values[m], x[n] + values[m]

            # 1. read the oldest sample and add the input
            line = MathTex(rf"y[{n}] = x[{n}] + y[{n - M}] = {x[n]} + {old} = ", rf"{new}")
            line.scale(0.75).move_to([0, -0.85, 0])
            self.play(FadeIn(line) if calc is None else ReplacementTransform(calc, line),
                      Indicate(nums[m], color=anim.GOLD, scale_factor=1.25), run_time=0.55)
            calc = line

            # 2. overwrite the oldest sample with the newest, and output it
            written = DecimalNumber(new, num_decimal_places=0).scale(0.9).move_to(cells[m])
            stem = VGroup(Line(ax.c2p(n + 0.5, 0), ax.c2p(n + 0.5, new),
                               color=anim.RED, stroke_width=3),
                          Dot(ax.c2p(n + 0.5, new), radius=0.07, color=anim.RED))
            self.play(ReplacementTransform(calc[1].copy(), written), FadeOut(nums[m]),
                      FadeIn(stem, shift=0.2 * UP), run_time=0.55)
            nums[m] = written
            values[m] = new
            stems.add(stem)

            # 3. advance the pointer, wrapping around at the end
            nxt = (m + 1) % M
            shift = (cells[nxt].get_center()[0] - cells[m].get_center()[0]) * RIGHT
            moves = [pointer.animate.shift(shift), hl.animate.move_to(cells[nxt])]
            if nxt == 0:
                moves.append(Indicate(wrap, color=anim.GOLD, scale_factor=1.05))
            self.play(*moves, run_time=0.45)

        self.wait(1.5)
        # reset, so the loop restarts cleanly
        zeros = [DecimalNumber(0, num_decimal_places=0).scale(0.9).move_to(c) for c in cells]
        self.play(FadeOut(stems), FadeOut(calc),
                  *[ReplacementTransform(v, z) for v, z in zip(nums, zeros)], run_time=0.8)
        self.wait(0.3)


anim.show(CircularBuffer)